<h1> Evaluate Conversation using Gemini 2.5 pro</h1>


In [ ]:
import pandas as pd
import google.generativeai as genai
import json
import time
import os
from tqdm import tqdm

# --- Configuration ---
# IMPORTANT: Set your Google AI Studio API Key as an environment variable
API_KEY = "YOUR_API_KEY_HERE" # Replace with your actual key or use os.environ.get("GOOGLE_API_KEY")

# Choose the model to act as the judge
JUDGE_MODEL = 'gemini-1.5-pro-latest'

# Input files (now with subquestion_id)
QUESTIONS_FILE = 'test_questions.csv'
RESPONSES_FILE = 'generated_responses_c.csv'

# Output file
RESULTS_FILE = 'evaluation_results_conversational.csv'

# --- The Judge Prompt (Updated for Conversations) ---
# The prompt is updated to accept a full conversation history.
JUDGE_PROMPT_TEMPLATE = """
You are an expert AI evaluator. Your task is to rigorously assess the quality of a chatbot's entire conversation with a user regarding a mental health query. Please evaluate the provided conversation based on the user's initial question and the chatbot's follow-up responses, using the six criteria below. For each criterion, provide a score from 1 to 5. Finally, provide a brief justification for your scores.

Your final output must be ONLY the structured JSON object. Do not add any conversational text, code formatting ticks, or anything else before or after the JSON.

**### EVALUATION CRITERIA ###**
1.  **Empathy and Tone (1-5):** Throughout the conversation, how empathetic, compassionate, and appropriate was the tone? (1=Cold/Robotic, 5=Very empathetic and warm)
2.  **Relevance of the Story (1-5):** How relevant and insightful was the mythological story to the user's core problem? (1=Irrelevant, 5=Highly insightful)
3.  **Clarity of the Core Teaching (1-5):** How clearly was the wisdom from the story explained and connected to the user's situation during the conversation? (1=Unclear, 5=Very clear and well-connected)
4.  **Actionability of Suggestions (1-5):** How clear, practical, and actionable were the suggested strategies provided in the conversation? (1=Vague/Impractical, 5=Very clear and actionable)
5.  **Justification of Suggestions (1-5):** How well were the suggested strategies logically connected to the mythological story? (1=No connection, 5=Strong, insightful connection)
6.  **Overall Perceived Helpfulness (1-5):** Considering the entire conversation, how helpful was this interaction likely to be for the user? (1=Not helpful, 5=Extremely helpful)

**### CONVERSATION TO EVALUATE ###**
{conversation_history}

**### REQUIRED OUTPUT FORMAT ###**
{{
  "scores": {{
    "empathy_and_tone": <score_1_to_5>,
    "relevance_of_story": <score_1_to_5>,
    "clarity_of_teaching": <score_1_to_5>,
    "actionability_of_suggestions": <score_1_to_5>,
    "justification_of_suggestions": <score_1_to_5>,
    "overall_helpfulness": <score_1_to_5>
  }},
  "justification": "<Your brief, one-paragraph reasoning for the scores>"
}}
"""

def evaluate_conversation(conversation_history, model):
    """
    Sends a single evaluation request for a full conversation to the Gemini API.
    Includes error handling and retries.
    """
    prompt = JUDGE_PROMPT_TEMPLATE.format(conversation_history=conversation_history)

    for attempt in range(3): # Retry up to 3 times
        try:
            api_response = model.generate_content(prompt)
            cleaned_response = api_response.text.strip().replace('```json', '').replace('```', '')
            return json.loads(cleaned_response)
        except (json.JSONDecodeError, AttributeError, ValueError) as e:
            print(f"  - Error processing response on attempt {attempt+1}: {e}. Retrying...")
            time.sleep(5)
        except Exception as e:
            print(f"  - An unexpected API error occurred on attempt {attempt+1}: {e}. Retrying...")
            time.sleep(10)

    print("  - Failed to get a valid JSON response after 3 attempts.")
    return None

def main():
    """
    Main function to run the evaluation pipeline for conversational responses.
    """
    if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
        print("ERROR: API_KEY not set. Please replace 'YOUR_API_KEY_HERE' with your actual key.")
        return
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(JUDGE_MODEL)

    try:
        df_questions = pd.read_csv(QUESTIONS_FILE)
        df_responses = pd.read_csv(RESPONSES_FILE)
    except FileNotFoundError as e:
        print(f"ERROR: Input file not found - {e}.")
        return

    # Merge dataframes and ensure subquestion_id is numeric for proper sorting
    df_merged = pd.merge(df_questions, df_responses, on=['question_id', 'subquestion_id'])
    df_merged['subquestion_id'] = pd.to_numeric(df_merged['subquestion_id'])

    all_results = []
    
    # Group by 'question_id' to process each conversation separately
    grouped = df_merged.groupby('question_id')

    for question_id, group in tqdm(grouped, total=len(grouped), desc="Evaluating Conversations"):
        print(f"\nProcessing Conversation ID: {question_id}...")
        
        # Sort the conversation turns by subquestion_id
        group = group.sort_values('subquestion_id')
        
        # Reconstruct the conversation history string
        conversation_history = ""
        for index, row in group.iterrows():
            conversation_history += f"User: {row['question_text']}\n"
            conversation_history += f"Chatbot: {row['response_c']}\n\n"
            
        evaluation = evaluate_conversation(conversation_history, model)
        
        if evaluation:
            flat_result = {
                'question_id': question_id,
                'system': 'System C',
                'justification': evaluation.get('justification', ''),
                **evaluation.get('scores', {})
            }
            all_results.append(flat_result)
        
        time.sleep(2) # Wait 2 seconds between each API call

    df_results = pd.DataFrame(all_results)
    df_results.to_csv(RESULTS_FILE, index=False)
    
    print(f"\nEvaluation complete! Results saved to '{RESULTS_FILE}'.")

if __name__ == "__main__":
    main()

<h1>Evalutate Conversation using ollama<h1>


In [17]:
import pandas as pd
import ollama
import json
import time
import os
from tqdm import tqdm

# --- Configuration ---
# Set the model to act as the judge, using an Ollama model name
JUDGE_MODEL = 'qwen3:32b' 

# IMPORTANT: Configure the Ollama client to connect to your Docker container.
# If you are running this script on your host machine and used the standard
# docker run command, 'http://localhost:11434' is correct.
# If your script is in another docker container on the same network,
# you might use 'http://ollama:11434' (where 'ollama' is the container name).
client = ollama.Client(host='http://localhost:11434')

# Input files
QUESTIONS_FILE = 'test_questions.csv'
RESPONSES_FILE = 'baseline_responses_cleaned.csv'

# Output file
RESULTS_FILE = 'evaluation_results_baseline_conversational_ollama.csv'

# --- The Judge Prompt (Unchanged) ---
JUDGE_PROMPT_TEMPLATE = """
You are a Critical Reviewer for a top-tier AI research conference. Your task is to provide a rigorous, critical, and nuanced assessment of a chatbot's response to a user's mental health query.
Your task is to rigorously assess the quality of a chatbot's entire conversation with a user regarding a mental health query. Please evaluate the provided conversation based on the user's initial question and the chatbot's follow-up responses, using the six criteria below. For each criterion, provide a score from 1 to 5. Finally, provide a brief justification for your scores.

**Primary Directive:** Your default mindset should be skeptical. A score of 5 should be exceptionally rare and reserved for responses that are flawless and profound. A competent, good response should score a 4. An average response is a 3. Do not hesitate to use scores of 1, 2, or 3 if the response has weaknesses.

Your final output must be ONLY the structured JSON object. Do not add any conversational text or anything else before or after the JSON.

**### EVALUATION CRITERIA ###**
1.  **Empathy and Tone (1-5):** Throughout the conversation, how empathetic, compassionate, and appropriate was the tone? (1=Cold/Robotic, 5=Very empathetic and warm)
2.  **Relevance of the Story (1-5):** How relevant and insightful was the mythological story to the user's core problem? (1=Irrelevant, 5=Highly insightful)
3.  **Clarity of the Core Teaching (1-5):** How clearly was the wisdom from the story explained and connected to the user's situation during the conversation? (1=Unclear, 5=Very clear and well-connected)
4.  **Actionability of Suggestions (1-5):** How clear, practical, and actionable were the suggested strategies provided in the conversation? (1=Vague/Impractical, 5=Very clear and actionable)
5.  **Justification of Suggestions (1-5):** How well were the suggested strategies logically connected to the mythological story? (1=No connection, 5=Strong, insightful connection)
6.  **Overall Perceived Helpfulness (1-5):** Considering the entire conversation, how helpful was this interaction likely to be for the user? (1=Not helpful, 5=Extremely helpful)

**### CONVERSATION TO EVALUATE ###**
{conversation_history}

**### REQUIRED OUTPUT FORMAT ###**
{{
  "scores": {{
    "empathy_and_tone": <score_1_to_5>,
    "relevance_of_story": <score_1_to_5>,
    "clarity_of_teaching": <score_1_to_5>,
    "actionability_of_suggestions": <score_1_to_5>,
    "justification_of_suggestions": <score_1_to_5>,
    "overall_helpfulness": <score_1_to_5>
  }},
  "justification": "<Your brief, one-paragraph reasoning for the scores>"
}}
"""

def evaluate_conversation_ollama(conversation_history, model_name, ollama_client):
    """
    Sends a single evaluation request to a specific Ollama client instance.
    Includes error handling and retries.
    """
    prompt = JUDGE_PROMPT_TEMPLATE.format(conversation_history=conversation_history)
    
    for attempt in range(3):
        try:
            # Use the provided client object to make the chat request
            response = ollama_client.chat(
                model=model_name,
                messages=[{'role': 'user', 'content': prompt}],
                format='json'
            )
            return json.loads(response['message']['content'])
        except json.JSONDecodeError as e:
            print(f"  - Error decoding JSON on attempt {attempt+1}: {e}. Retrying...")
            time.sleep(5)
        except Exception as e:
            print(f"  - An unexpected Ollama API error occurred on attempt {attempt+1}: {e}. Retrying...")
            time.sleep(10)

    print(f"  - Failed to get a valid response from Ollama after 3 attempts.")
    return None

def main():
    """
    Main function to run the evaluation pipeline using a specific Ollama client.
    """
    print(f"Using Ollama model '{JUDGE_MODEL}' via client at {client._client.base_url}")
    
    try:
        df_questions = pd.read_csv(QUESTIONS_FILE)
        df_responses = pd.read_csv(RESPONSES_FILE)
    except FileNotFoundError as e:
        print(f"ERROR: Input file not found - {e}.")
        return

    df_merged = pd.merge(df_questions, df_responses, on=['question_id', 'subquestion_id'])
    df_merged['subquestion_id'] = pd.to_numeric(df_merged['subquestion_id'])

    all_results = []
    
    grouped = df_merged.groupby('question_id')

    for question_id, group in tqdm(grouped, total=len(grouped), desc="Evaluating Conversations with Ollama"):
        print(f"\nProcessing Conversation ID: {question_id}...")
        
        group = group.sort_values('subquestion_id')
        
        conversation_history = ""
        for index, row in group.iterrows():
            conversation_history += f"User: {row['question_text']}\n"
            conversation_history += f"Chatbot: {row['response_c']}\n\n"
            
        # Pass the model name and the client instance to the evaluation function
        evaluation = evaluate_conversation_ollama(conversation_history, JUDGE_MODEL, client)
        
        if evaluation:
            # Ensure the nested 'scores' dictionary exists before trying to unpack it
            scores = evaluation.get('scores', {})
            flat_result = {
                'question_id': question_id,
                'system': 'System C (Ollama)',
                'justification': evaluation.get('justification', ''),
                **scores
            }
            all_results.append(flat_result)
        
        time.sleep(1) 

    df_results = pd.DataFrame(all_results)
    
    # Define the desired column order for the CSV file
    column_order = [
        'question_id', 'system', 'empathy_and_tone', 'relevance_of_story',
        'clarity_of_teaching', 'actionability_of_suggestions', 
        'justification_of_suggestions', 'overall_helpfulness', 'justification'
    ]
    
    # Reorder DataFrame columns and handle missing columns gracefully
    df_results = df_results.reindex(columns=column_order)

    df_results.to_csv(RESULTS_FILE, index=False)
    
    print(f"\nEvaluation complete! Results saved to '{RESULTS_FILE}'.")

if __name__ == "__main__":
    main()

Using Ollama model 'qwen3:32b' via client at http://localhost:11434


Evaluating Conversations with Ollama:   0%|          | 0/40 [00:00<?, ?it/s]


Processing Conversation ID: 1...


Evaluating Conversations with Ollama:   2%|▎         | 1/40 [00:31<20:24, 31.39s/it]


Processing Conversation ID: 2...


...


Processing Conversation ID: 39...


Evaluating Conversations with Ollama:  98%|█████████▊| 39/40 [11:25<00:17, 17.08s/it]


Processing Conversation ID: 40...


Evaluating Conversations with Ollama: 100%|██████████| 40/40 [11:41<00:00, 17.55s/it]


Evaluation complete! Results saved to 'evaluation_results_baseline_conversational_ollama.csv'.


<h1>Generate responses using baseline</h1>

In [15]:
import pandas as pd
import ollama
import json
import time
from tqdm import tqdm

# --- Configuration ---
# Set the model to generate responses, using an Ollama model name
BASE_MODEL = 'qwen3:32b' 

# IMPORTANT: Configure the Ollama client to connect to your Docker container.
# If you are running this script on your host machine and used the standard
# docker run command, 'http://localhost:11434' is the correct address.
client = ollama.Client(host='http://localhost:11434')

# Input file (must contain 'question_id' and 'subquestion_id')
QUESTIONS_FILE = 'test_questions.csv'

# Output file for the generated conversational responses
OUTPUT_FILE = 'generated_responses_a_ollama.csv'

# --- The System Prompt ---
# This sets the persona for the chatbot.
BASELINE_SYSTEM_PROMPT = "You are a helpful and supportive AI assistant."

def generate_ollama_response(conversation_history, model_name, ollama_client):
    """
    Sends a request to the Ollama API with the current conversation history.
    Includes error handling and retries.
    
    Args:
        conversation_history (list): A list of message dictionaries for the API.
        model_name (str): The name of the Ollama model to use.
        ollama_client (ollama.Client): The Ollama client instance.

    Returns:
        str: The generated text response from the model.
    """
    for attempt in range(3): # Retry up to 3 times
        try:
            # The entire conversation history is sent to the model
            response = ollama_client.chat(
                model=model_name,
                messages=conversation_history
            )
            # Extract the response content and strip any whitespace
            return response['message']['content'].strip()
        except Exception as e:
            print(f"  - An unexpected API error occurred on attempt {attempt+1}: {e}. Retrying...")
            time.sleep(10)

    print("  - Failed to get a valid response after 3 attempts.")
    return "Error: Failed to generate response."

def main():
    """
    Main function to run the conversational response generation pipeline.
    """
    print(f"Using Ollama model '{BASE_MODEL}' to generate responses.")
    
    # Load the input questions
    try:
        df_questions = pd.read_csv(QUESTIONS_FILE)
        # Ensure subquestion_id is present for conversational logic
        if 'subquestion_id' not in df_questions.columns:
            print(f"ERROR: The file '{QUESTIONS_FILE}' must contain a 'subquestion_id' column for conversational generation.")
            return
    except FileNotFoundError as e:
        print(f"ERROR: Input file not found - {e}. Please ensure '{QUESTIONS_FILE}' is in the same folder.")
        return

    all_results = []
    
    # Group by 'question_id' to process each conversation independently
    grouped = df_questions.groupby('question_id')

    for question_id, group in tqdm(grouped, total=len(grouped), desc="Generating Conversational Responses"):
        print(f"\nProcessing Conversation ID: {question_id}...")
        
        # Initialize the history for this specific conversation with the system prompt
        conversation_history = [{'role': 'system', 'content': BASELINE_SYSTEM_PROMPT}]
        
        # Sort the turns of the conversation by subquestion_id
        group = group.sort_values('subquestion_id')
        
        # Iterate through each turn in the current conversation
        for index, row in group.iterrows():
            # 1. Add the user's current question to the history
            user_message = {'role': 'user', 'content': row['question_text']}
            conversation_history.append(user_message)
            
            # 2. Generate a response based on the entire history up to this point
            assistant_response = generate_ollama_response(conversation_history, BASE_MODEL, client)
            
            # 3. Add the model's response to the history for context in the next turn
            assistant_message = {'role': 'assistant', 'content': assistant_response}
            conversation_history.append(assistant_message)
            
            # 4. Store the result for this specific turn
            result = {
                'question_id': row['question_id'],
                'subquestion_id': row['subquestion_id'],
                'response_a': assistant_response
            }
            all_results.append(result)
            
            # Optional: A small delay between requests
            time.sleep(1)

    # Convert the list of results into a DataFrame and save to CSV
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(OUTPUT_FILE, index=False)
    
    print(f"\nGeneration complete! Conversational responses saved to '{OUTPUT_FILE}'.")

if __name__ == "__main__":
    main()

Using Ollama model 'qwen3:32b' to generate responses.


Generating Conversational Responses:   0%|          | 0/40 [00:00<?, ?it/s]


Processing Conversation ID: 1...


Generating Conversational Responses:   2%|▎         | 1/40 [03:05<2:00:39, 185.62s/it]


Processing Conversation ID: 2...


...


Processing Conversation ID: 39...


Generating Conversational Responses:  98%|█████████▊| 39/40 [2:00:21<03:10, 190.52s/it]


Processing Conversation ID: 40...


Generating Conversational Responses: 100%|██████████| 40/40 [2:03:47<00:00, 185.69s/it]


Generation complete! Conversational responses saved to 'generated_responses_a_ollama.csv'.


<h1>LLM vs Human score evaluation</h1>

In [18]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score
import numpy as np
from tabulate import tabulate
from itertools import combinations

# --- Configuration ---
# Input files updated to your new conversational Ollama results and 5-psychologist evaluation file
LLM_JUDGE_RESULTS_FILE = 'evaluation_results_conversational_ollama.csv'
HUMAN_RESULTS_FILE = 'Final_human_evaluations_improved.csv'

def calculate_tolerant_agreement(ratings1, ratings2, tolerance=1):
    """Calculates the percentage of agreement within a given tolerance."""
    if len(ratings1) != len(ratings2):
        return 0.0
    
    agree_count = sum(1 for r1, r2 in zip(ratings1, ratings2) if abs(r1 - r2) <= tolerance)
    return (agree_count / len(ratings1)) * 100

def calculate_agreement(df_merged, criterion_name):
    """
    Calculates agreement metrics for a single criterion, now for 5 human raters.
    """
    # Define column names
    llm_col = criterion_name
    human_cols = [f'psychologist_{i}_{criterion_name}' for i in range(1, 6)]

    # Extract the rating lists, dropping any rows with missing data for the relevant columns
    df_filtered = df_merged[[llm_col] + human_cols].dropna()
    
    if len(df_filtered) < 2:
        print(f"  - Warning: Not enough valid data for criterion '{criterion_name}'. Skipping.")
        return np.nan, np.nan, np.nan, np.nan

    llm_ratings = df_filtered[llm_col].astype(int).tolist()
    
    # --- Calculate average pairwise Kappa between the five human psychologists ---
    human_kappa_scores = []
    for i, j in combinations(range(1, 6), 2):
        p1_ratings = df_filtered[f'psychologist_{i}_{criterion_name}'].astype(int).tolist()
        p2_ratings = df_filtered[f'psychologist_{j}_{criterion_name}'].astype(int).tolist()
        kappa = cohen_kappa_score(p1_ratings, p2_ratings, weights='linear')
        human_kappa_scores.append(kappa)
    
    avg_human_kappa = np.mean(human_kappa_scores) if human_kappa_scores else np.nan

    # --- Create an average human score (from 5 raters) for comparison with the LLM ---
    avg_human_ratings = df_filtered[human_cols].mean(axis=1).round().astype(int).tolist()

    # --- Calculate agreement between the LLM and the average human score ---
    llm_vs_human_kappa = cohen_kappa_score(llm_ratings, avg_human_ratings, weights='linear')
    llm_vs_human_percent_agreement_strict = accuracy_score(llm_ratings, avg_human_ratings) * 100
    llm_vs_human_percent_agreement_tolerant = calculate_tolerant_agreement(llm_ratings, avg_human_ratings, tolerance=1)
    
    return avg_human_kappa, llm_vs_human_kappa, llm_vs_human_percent_agreement_strict, llm_vs_human_percent_agreement_tolerant

def interpret_kappa(kappa_score):
    """Provides a qualitative interpretation of a Kappa score."""
    if pd.isna(kappa_score): return "N/A"
    if kappa_score > 0.80: return "Almost Perfect"
    if kappa_score > 0.60: return "Substantial"
    if kappa_score > 0.40: return "Moderate"
    if kappa_score > 0.20: return "Fair"
    return "Slight"

def main():
    """
    Main function to load data and run the full agreement analysis for 5 raters.
    """
    try:
        df_llm = pd.read_csv(LLM_JUDGE_RESULTS_FILE)
        df_human = pd.read_csv(HUMAN_RESULTS_FILE)
    except FileNotFoundError as e:
        print(f"ERROR: Input file not found - {e}.")
        return

    # --- Data Cleaning and Preparation ---
    df_llm.columns = df_llm.columns.str.strip()
    df_human.columns = df_human.columns.str.strip()

    # Fix typo in human evaluation file for successful merging
    df_human = df_human.rename(columns=lambda c: c.replace('perceoved', 'perceived'))
    # Assuming 'overall_perceived_helpfulness' in human file should match 'overall_helpfulness' in LLM file
    df_human = df_human.rename(columns=lambda c: c.replace('perceived_helpfulness', 'helpfulness'))


    df_llm['question_id'] = df_llm['question_id'].astype(str).str.strip()
    df_human['question_id'] = df_human['question_id'].astype(str).str.strip()
    
    criteria_short_names = [
        'empathy_and_tone', 'relevance_of_story', 'clarity_of_teaching',
        'actionability_of_suggestions', 'justification_of_suggestions', 'overall_helpfulness'
    ]
    
    df_merged = pd.merge(df_llm, df_human, on='question_id', how='inner')
    
    if df_merged.empty:
        print("ERROR: Merge failed. No matching 'question_id's found. Check column names and values.")
        return
        
    print(f"--- Data Merge Successful. Found {len(df_merged)} matching rows. ---")
    print("-" * 80)
    print("Inter-Rater Reliability and Agreement Analysis (LLM vs. 5 Humans)")
    print("-" * 80)

    results = []
    for criterion in criteria_short_names:
        h_kappa, llm_kappa, llm_pa_strict, llm_pa_tolerant = calculate_agreement(df_merged, criterion)
        results.append({
            'Criterion': criterion.replace('_', ' ').title(),
            'Human vs Human (Avg. κ)': f"{h_kappa:.2f}",
            'LLM vs Human (κ)': f"{llm_kappa:.2f}",
            'LLM Kappa Level': interpret_kappa(llm_kappa),
            'LLM vs Human (% Agree Strict)': f"{llm_pa_strict:.1f}%",
            'LLM vs Human (% Agree +/-1)': f"{llm_pa_tolerant:.1f}%"
        })

    df_results = pd.DataFrame(results)
    print(tabulate(df_results, headers='keys', tablefmt='grid', showindex=False))
    print("-" * 80)
    print("\nNotes:")
    print("  - 'Human vs Human (Avg. κ)' is the average of pairwise Cohen's Kappa scores for all 10 pairs of human raters.")
    print("  - 'LLM vs Human (κ)' compares the LLM's ratings to the rounded average of the 5 human raters.")

if __name__ == "__main__":
    main()

--- Data Merge Successful. Found 40 matching rows. ---
--------------------------------------------------------------------------------
Inter-Rater Reliability and Agreement Analysis (LLM vs. 5 Humans)
--------------------------------------------------------------------------------


KeyError: "['psychologist_1_relevance_of_story', 'psychologist_2_relevance_of_story', 'psychologist_3_relevance_of_story', 'psychologist_4_relevance_of_story', 'psychologist_5_relevance_of_story'] not in index"